In [11]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from agents import Agent, Runner, trace, function_tool
from typing import Dict
import asyncio
import gradio as gr
import json
# ruff: noqa: F704

In [12]:
load_dotenv(override=True)

True

In [13]:
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [8]:
MODEL = "gpt-4o-mini"
client = OpenAI()

In [6]:
open_router_api_key = os.getenv('OPENROUTER_API_KEY')

if not open_router_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not open_router_api_key.startswith("sk-or-v1-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif open_router_api_key.strip() != open_router_api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [7]:
client= OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_router_api_key
)
MODEL = "gpt-4o-mini"

In [14]:
requesty_api_key = os.getenv('REQUESTY_API_KEY')

if not requesty_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not requesty_api_key.startswith("sk-or-v1-"):
    print("An API key was found, but it doesn't start sk-; please check you're using the right key - see troubleshooting notebook")
elif requesty_api_key.strip() != requesty_api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

An API key was found, but it doesn't start sk-; please check you're using the right key - see troubleshooting notebook


In [15]:
client= OpenAI(
    base_url="https://router.requesty.ai/v1",
    api_key=requesty_api_key
)
MODEL = "gpt-4o-mini"

In [ ]:
# MODEL = "llama3.2"
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [17]:
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

## Types of prompts


Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**System prompt** -- tells them what task they are performing and what tone they should use

**User prompt** -- the conversation starter that they should reply to 


## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)


In [18]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [19]:
# To give you a preview -- calling OpenAI with system and user messages:

response = client.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

APIConnectionError: Connection error.

## TOOLS

In [14]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += ("Always be accurate. If you don't know the answer, say so. Do not make up an answer "
                  "or attempt to use previous knowledge")

In [15]:
print(system_message)

You are a helpful assistant for an Airline called FlightAI. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you don't know the answer, say so. Do not make up an answer or attempt to use previous knowledge


In [ ]:
# This function looks rather simpler than the one from my video, because we're taking advantage of the latest Gradio updates
def chat(message, history):
    print("\n💬 New message:", message)
    print("🧠 History:", history)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages",js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.



💬 New message: Hello, what is the ticket price to Berlin?
🧠 History: []

💬 New message: Hello
🧠 History: []

💬 New message: I know
🧠 History: [{'role': 'user', 'metadata': None, 'content': 'Hello', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you today?', 'options': None}]

💬 New message: What is your name
🧠 History: [{'role': 'user', 'metadata': None, 'content': 'Hello', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you today?', 'options': None}, {'role': 'user', 'metadata': None, 'content': 'I know', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Great! If you have any questions or need assistance, feel free to ask.', 'options': None}]

💬 New message: Give me price for berlin
🧠 History: [{'role': 'user', 'metadata': None, 'content': 'Hello', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you today?', 'options': None}, 

In [ ]:
# This function looks rather simpler than the one from my video, because we're taking advantage of the latest Gradio updates
def chat(message):
    print("\n💬 New message:", message)
    messages = [{"role": "system", "content": system_message}]  + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages",js=force_dark_mode).launch()

/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/utils.py:1038: UserWarning: Expected 1 arguments for function <function chat at 0x125135f80>, received 2.
  warnings.warn(
/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/utils.py:1046: UserWarning: Expected maximum 1 arguments for function <function chat at 0x125135f80>, received 2.
  warnings.warn(
/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/utils.py:1038: UserWarning: Expected 1 arguments for function <function chat at 0x125136020>, received 2.
  warnings.warn(
/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/utils.py:1046: UserWarning: Expected maximum 1 arguments for function <function chat at 0x125136020>, received 2.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2220, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1729, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ank/techspace/ai/llm_agents/.venv/lib/python3.12/site-packages/gradio/utils.p

In [55]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [56]:
get_ticket_price("Berlin")

Tool get_ticket_price called for Berlin


'$499'

In [57]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [58]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        print("OpenAI Says: ", response.choices[0].finish_reason)
        message = response.choices[0].message
        print("OpenAI Says How to: ", response.choices[0].message)
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        print("Messages after tool call: ", messages)
        response = client.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [60]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    print("Tool Call is: ", tool_call)
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get('destination_city')
    price = get_ticket_price(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"price": price}),
        "tool_call_id": tool_call.id
    }
    return response, city

In [61]:
gr.ChatInterface(fn=chat, type="messages",js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


OpenAI Says:  tool_calls
OpenAI Says How to:  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_AEIPr9LSFRINRQyXGQN0mk0m', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')])
Tool Call is:  ChatCompletionMessageToolCall(id='call_AEIPr9LSFRINRQyXGQN0mk0m', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')
Tool get_ticket_price called for Berlin
Messages after tool call:  [{'role': 'system', 'content': "You are a helpful assistant for an Airline called FlightAI. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you don't know the answer, say so. Do not make up an answer to attempt to use previous knowledge"}, {'role': 'user', 'content': 'How much is the ticket price to Berlin?'}, ChatCompletionMessage(content=None, refusal=None, role='ass

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] +  [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        print("OpenAI Says: ", response.choices[0].finish_reason)
        message = response.choices[0].message
        print("OpenAI Says How to: ", response.choices[0].message)
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        print("Messages after tool call: ", messages)
        response = client.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [63]:
gr.ChatInterface(fn=chat, type="messages",js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


OpenAI Says:  tool_calls
OpenAI Says How to:  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_warG5tWXz9HunA5VmsyWtSkf', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')])
Tool Call is:  ChatCompletionMessageToolCall(id='call_warG5tWXz9HunA5VmsyWtSkf', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')
Tool get_ticket_price called for Berlin
Messages after tool call:  [{'role': 'system', 'content': "You are a helpful assistant for an Airline called FlightAI. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you don't know the answer, say so. Do not make up an answer to attempt to use previous knowledge"}, {'role': 'user', 'content': 'How much is the ticket price to Berlin?'}, ChatCompletionMessage(content=None, refusal=None, role='ass

In [47]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
@function_tool
def get_ticket_price(destination_city: str) -> str:
    """Get the price of a return ticket to the destination city. 
    Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'
    """
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [31]:
print(get_ticket_price)

FunctionTool(name='get_ticket_price', description="Get the price of a return ticket to the destination city. \nCall this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'", params_json_schema={'properties': {'destination_city': {'title': 'Destination City', 'type': 'string'}}, 'required': ['destination_city'], 'title': 'get_ticket_price_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x123518ea0>, strict_json_schema=True, is_enabled=True)


In [32]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [33]:
agent = Agent(
    name="FlightAI",
    instructions=(
        system_message
    ),
    tools=[get_ticket_price],
    model="gpt-4o-mini",
)
result = await (Runner.run(agent, input="How much is the ticket price to Berlin"))
print(result.final_output)

Tool get_ticket_price called for Berlin
The ticket price to Berlin is $499.


In [34]:
async def chat_async(message, history):
    result = await Runner.run(agent, message)
    return result.final_output

In [35]:
gr.ChatInterface(fn=chat_async, type="messages",js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


Tool get_ticket_price called for Berlin


In [36]:
async def chat_async(message, history):
    print(history)
    print(message)
    items = [{"role":m.get("role"),"content":m.get("content")} for m in history if isinstance(m.get("content"), str)]
    items.append({"role": "user", "content": message})
    print(items)
    result = await Runner.run(agent, items)
    return result.final_output

In [37]:
gr.ChatInterface(fn=chat_async, type="messages",js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


[]
How much is the ticket price for Berlin?
[{'role': 'user', 'content': 'How much is the ticket price for Berlin?'}]
Tool get_ticket_price called for Berlin
[{'role': 'user', 'metadata': None, 'content': 'How much is the ticket price for Berlin?', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'The ticket price to Berlin is $499.', 'options': None}]
Which city price did i just ask?
[{'role': 'user', 'content': 'How much is the ticket price for Berlin?'}, {'role': 'assistant', 'content': 'The ticket price to Berlin is $499.'}, {'role': 'user', 'content': 'Which city price did i just ask?'}]


## MCP

In [13]:
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

In [14]:
params = {"command": "uv", "args": ["run", "mcp_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

In [15]:
mcp_tools

[Tool(name='get_ticket_price', description="Get the price of a return ticket to the destination city. \n    Call this whenever you need to know the ticket price, for example when a customer\n    asks 'How much is a ticket to this city'\n    ", inputSchema={'properties': {'destination_city': {'title': 'Destination City', 'type': 'string'}}, 'required': ['destination_city'], 'title': 'get_ticket_priceArguments', 'type': 'object'}, annotations=None),
 Tool(name='convert_usd_to_eur', description='\n    Convert a USD price like "$499" to EUR using the provided USD->EUR rate.\n    Returns a string like "€459.08".\n    ', inputSchema={'properties': {'price_usd': {'title': 'Price Usd', 'type': 'string'}, 'rate': {'default': 0.92, 'title': 'Rate', 'type': 'number'}}, 'required': ['price_usd'], 'title': 'convert_usd_to_eurArguments', 'type': 'object'}, annotations=None)]

In [17]:
params = {"command": "uv", "args": ["run", "mcp_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    price = await server.call_tool(tool_name="get_ticket_price", arguments={"destination_city": "Berlin"})

In [18]:
price

CallToolResult(meta=None, content=[TextContent(type='text', text='$499', annotations=None)], isError=False)

In [23]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from contextlib import AsyncExitStack
from mcp.types import TextContent

In [105]:
server_params = StdioServerParameters(command="uv", args=["run", "mcp_server.py"])
async with AsyncExitStack() as stack:
    try:
        # Establish a stdio connection to the server using the server parameters
        transport = await stack.enter_async_context(stdio_client(server_params))
        read, write, *_ = transport
        # Create a client session using the read and write streams from the connection
        session = await stack.enter_async_context(ClientSession(read, write))
        # Initialize the session (e.g., perform handshake or setup operations)
        await session.initialize()

        # Load the MCP tools from the connected server using the adapter function
        response = await session.list_tools()

        # Iterate over each tool and add it to the aggregated tools list
        tools = response.tools
        print("\n✅ List of tools")
        for tool in tools:
            print(tool)
        print("\n✅ Connected to server with tools:", [tool.name for tool in tools])

    except Exception as e:
      # Handle any errors that occur during connection or tool loading for the server
      print(f"❌ Failed to connect to server : {e}")

    # call_tool: get_ticket_price
    price_res = await session.call_tool(
        name="get_ticket_price",
        arguments={"destination_city": "Berlin"},
    )
    price_text = next((c.text for c in (price_res.content or []) if isinstance(c, TextContent)), None)
    print("Received Response from tool get_ticket_price('Berlin') ->", price_text or price_res)

    #call_tool: convert_usd_to_eur (optional, if your server implements it)
    
    eur_res = await session.call_tool(
                name="convert_usd_to_eur",
                arguments={"price_usd": price_text, "rate": 0.92},
                )
    eur_text = next((c.text for c in (eur_res.content or []) if isinstance(c, TextContent)), None)
    print("Received Response from toolconvert_usd_to_eur('$499', 0.92) ->", eur_text or eur_res)
    


✅ List of tools
name='get_ticket_price' description="Get the price of a return ticket to the destination city. \n    Call this whenever you need to know the ticket price, for example when a customer\n    asks 'How much is a ticket to this city'\n    " inputSchema={'properties': {'destination_city': {'title': 'Destination City', 'type': 'string'}}, 'required': ['destination_city'], 'title': 'get_ticket_priceArguments', 'type': 'object'} annotations=None
name='convert_usd_to_eur' description='\n    Convert a USD price like "$499" to EUR using the provided USD->EUR rate.\n    Returns a string like "€459.08".\n    ' inputSchema={'properties': {'price_usd': {'title': 'Price Usd', 'type': 'string'}, 'rate': {'default': 0.92, 'title': 'Rate', 'type': 'number'}}, 'required': ['price_usd'], 'title': 'convert_usd_to_eurArguments', 'type': 'object'} annotations=None

✅ Connected to server with tools: ['get_ticket_price', 'convert_usd_to_eur']
Received Response from tool get_ticket_price('Berlin'

In [84]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from contextlib import AsyncExitStack
from mcp.types import TextContent
from agents.mcp import MCPServerStdio

In [106]:
server_params = [{"command": "uv", "args":["run", "mcp_server.py"]}]
async with AsyncExitStack() as stack:
    # build & connect servers (RETURN actual server objects, not coroutines)
    mcp_servers = [
        await stack.enter_async_context(
            MCPServerStdio(
                params=p,
                name=f"stdio-{i}",
                client_session_timeout_seconds=120,
                # optional:
                # cache_tools_list=True,
                # use_structured_content=True,
            )
        )
        for i, p in enumerate(server_params)
    ]

    agent = Agent(
        name="FlightAI",
        instructions=(
            system_message
        ),
        #tools=[get_ticket_price],
        mcp_servers=mcp_servers,
        model="gpt-4o-mini",
        )
    result = await (Runner.run(agent, input="How much is the ticket price to Berlin. And convert it to Euros too, please."))
    print(result.final_output)

The ticket price to Berlin is $499, which converts to €459.08.


In [104]:
server_params = [{"command": "uv", "args":["run", "mcp_server.py"]}]
#async with AsyncExitStack() as stack:
stack = AsyncExitStack()
await stack.__aenter__() 
    # build & connect servers (RETURN actual server objects, not coroutines)
mcp_servers = [
    await stack.enter_async_context(
        MCPServerStdio(
            params=server_param,
            name=f"stdio-{i}",
            client_session_timeout_seconds=120,
            # optional:
            # cache_tools_list=True,
            # use_structured_content=True,
        )
    )
    for i, server_param in enumerate(server_params)
]

agent = Agent(
    name="FlightAI",
    instructions=(
        system_message
    ),
    #tools=[get_ticket_price],
    mcp_servers=mcp_servers,
    model="gpt-4o-mini",
    )
    #result = await (Runner.run(agent, input="How much is the ticket price to Berlin. And convert it to Euros too, please."))
    #print(result.final_output)
    
async def chat_mcp(message, history):
        print(history)
        print(message)
        items = [{"role":m.get("role"),"content":m.get("content")} for m in history if isinstance(m.get("content"), str)]
        items.append({"role": "user", "content": message})
        result = await Runner.run(agent, items)
        return result.final_output

gr.ChatInterface(fn=chat_mcp, type="messages",js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


[]
How much is the ticket price to berlin
[{'role': 'user', 'metadata': None, 'content': 'How much is the ticket price to berlin', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'The ticket price to Berlin is $499.', 'options': None}]
ok
[{'role': 'user', 'metadata': None, 'content': 'How much is the ticket price to berlin', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'The ticket price to Berlin is $499.', 'options': None}, {'role': 'user', 'metadata': None, 'content': 'ok', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'If you need any more information or assistance, feel free to ask!', 'options': None}]
how about india


In [107]:
await stack.aclose()

In [112]:
server_params = [{"command": "uv", "args":["run", "mcp_server.py"]}]
#async with AsyncExitStack() as stack:
async with AsyncExitStack() as stack:
    mcp_servers = [
        await stack.enter_async_context(
            MCPServerStdio(
                params=server_param,
                name=f"stdio-{i}",
                client_session_timeout_seconds=120,
                # optional:
                # cache_tools_list=True,
                # use_structured_content=True,
            )
        )
        for i, server_param in enumerate(server_params)
    ]
    
    agent = Agent(
        name="FlightAI",
        instructions=(
            system_message
        ),
        #tools=[get_ticket_price],
        mcp_servers=mcp_servers,
        model="gpt-4o-mini",
        )
        #result = await (Runner.run(agent, input="How much is the ticket price to Berlin. And convert it to Euros too, please."))
        #print(result.final_output)
        
    async def chat_mcp(message, history):
            print(history)
            print(message)
            items = [{"role":m.get("role"),"content":m.get("content")} for m in history if isinstance(m.get("content"), str)]
            items.append({"role": "user", "content": message})
            result = await Runner.run(agent, items)
            return result.final_output
    
    gr.ChatInterface(fn=chat_mcp, type="messages",js=force_dark_mode).launch()

    try:
        while True:
            await asyncio.sleep(3600)
    # except (KeyboardInterrupt, SystemExit):
    except:
        pass

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.
